https://zenodo.org/records/10695923?preview_file=carotid_bifurcation_database.zip

this is where the data is obtained from

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyvista as pv
import pyvistaqt as pvqt
import scipy.spatial as spy_spatial
import scipy.spatial.transform as spy_transform

In [ ]:
case_dir, side = "w_044", "right"
centerlines = pv.read(
  f"./crops/data/carotid/{case_dir}/{case_dir}_{side}_lumen_centerlines.vtp"
)
lumen = pv.read(f"./crops/data/carotid/{case_dir}/{case_dir}_{side}_lumen.stl")
radius = np.asarray(centerlines["MaximumInscribedSphereRadius"])
point_tree = spy_spatial.cKDTree(centerlines.points)

### Helpers

In [ ]:
# Every polyline is stored as [num_points, id_0, ..., id_{num_points - 1}].
def split_lines(centerlines):
  lines, cursor = [], 0
  while cursor < centerlines.lines.size:
    num_points = centerlines.lines[cursor]
    lines.append(centerlines.lines[cursor + 1 : cursor + 1 + num_points])
    cursor += num_points + 1
  return lines


def picked_lines(centerlines, lines, waypoint_ids, tol=1.0e-6):
  # Lines through every waypoint, or through any of them if no line covers all.
  # The trunk points are duplicated on every line that starts at the inlet.
  on_lines = [
    {
      index
      for index, point_ids in enumerate(lines)
      if np.linalg.norm(centerlines.points[point_ids] - point, axis=-1).min() < tol
    }
    for point in centerlines.points[waypoint_ids]
  ]
  return sorted(set.intersection(*on_lines) or set.union(*on_lines))


def waypoint_spline(centerlines, waypoint_ids, spacing):
  # Interpolating spline through the waypoints, uniform in arc length (mm).
  waypoints = centerlines.points[waypoint_ids].astype(float)
  keep = np.concatenate([[True], np.linalg.norm(np.diff(waypoints, axis=0), axis=-1) > 0])
  waypoints = waypoints[keep]
  dense = np.asarray(pv.Spline(waypoints, 100 * len(waypoints) + 1000).points)
  dense_arc = arc_length(dense)
  arc = np.linspace(0.0, dense_arc[-1], int(np.ceil(dense_arc[-1] / spacing)) + 1)
  return np.stack([np.interp(arc, dense_arc, dense[:, dim]) for dim in range(3)], axis=-1)


def arc_length(points):
  return np.concatenate(
    [[0.0], np.cumsum(np.linalg.norm(np.diff(points, axis=0), axis=-1))]
  )


def rotation_minimizing_frames(points):
  # Double reflection method (Wang et al. 2008), so the tube does not twist.
  tangents = np.gradient(points, axis=0)
  tangents /= np.linalg.norm(tangents, axis=-1, keepdims=True)
  helper = np.eye(3)[np.argmin(np.abs(tangents[0]))]
  normals = [np.cross(tangents[0], helper) / np.linalg.norm(np.cross(tangents[0], helper))]
  for i in range(len(points) - 1):
    v1 = points[i + 1] - points[i]
    c1 = v1 @ v1
    normal_l = normals[-1] - 2.0 / c1 * (v1 @ normals[-1]) * v1
    tangent_l = tangents[i] - 2.0 / c1 * (v1 @ tangents[i]) * v1
    v2 = tangents[i + 1] - tangent_l
    c2 = v2 @ v2
    normals.append(normal_l - 2.0 / c2 * (v2 @ normal_l) * v2 if c2 > 1e-20 else normal_l)
  normals = np.asarray(normals)
  return tangents, normals, np.cross(tangents, normals)


def annular_tube(path, inner_radius, outer_radius, num_sides=48):
  """Closed wall between two coaxial tubes along `path`, open at both ends."""
  _, normals, binormals = rotation_minimizing_frames(path)
  theta = np.linspace(0.0, 2.0 * np.pi, num_sides, endpoint=False)
  ring = np.einsum("nd,k->nkd", normals, np.cos(theta)) + np.einsum(
    "nd,k->nkd", binormals, np.sin(theta)
  )
  num_rings = len(path)
  outer = path[:, None] + outer_radius * ring
  inner = path[:, None] + inner_radius * ring
  points = np.concatenate([outer.reshape(-1, 3), inner.reshape(-1, 3)])

  def index(surface, i, k):
    return surface * num_rings * num_sides + i * num_sides + k % num_sides

  quads = []
  for i in range(num_rings - 1):
    for k in range(num_sides):
      quads.append([index(0, i, k), index(0, i, k + 1), index(0, i + 1, k + 1), index(0, i + 1, k)])
      quads.append([index(1, i, k), index(1, i + 1, k), index(1, i + 1, k + 1), index(1, i, k + 1)])
  for k in range(num_sides):  # annular faces at the two open ends
    quads.append([index(0, 0, k), index(1, 0, k), index(1, 0, k + 1), index(0, 0, k + 1)])
    last = num_rings - 1
    quads.append([index(0, last, k), index(0, last, k + 1), index(1, last, k + 1), index(1, last, k)])
  faces = np.column_stack([np.full(len(quads), 4), quads]).ravel()
  return pv.PolyData(points, faces).triangulate().compute_normals(auto_orient_normals=True)


def catheter_track(
  centerlines,
  waypoint_ids,
  inner_radius,
  outer_radius,
  spacing=0.25,
  num_sides=48,
  out_scale=1.0e-3,
):
  """Target curve through the picked waypoints and the annular tube around it.

  Lengths go in in mm (centerline units) and come out multiplied by `out_scale`.
  Both outputs share one rigid frame: the first waypoint (the mouth) at the
  origin with the insertion direction along +x.
  """
  path = waypoint_spline(centerlines, waypoint_ids, spacing)
  rotation, _ = spy_transform.Rotation.align_vectors(
    [[1.0, 0.0, 0.0]], [path[1] - path[0]]
  )
  track = out_scale * rotation.apply(path - path[0])
  target_curve = pd.DataFrame(
    np.column_stack([arc_length(track), track]), columns=["s", "x", "y", "z"]
  )

  # The swept rings fold over wherever the path bends tighter than the tube.
  chords = np.diff(path, axis=0)
  chord_length = np.linalg.norm(chords, axis=-1)
  chords /= chord_length[:, None]
  turn = np.arccos(np.clip(np.einsum("nd,nd->n", chords[:-1], chords[1:]), -1.0, 1.0))
  tightest_bend = np.min(chord_length[1:] / np.maximum(turn, 1e-12))
  if tightest_bend < outer_radius:
    print(f"WARNING  tightest bend {tightest_bend:.2f} mm < outer radius, the tube self-intersects")

  tube = annular_tube(track, out_scale * inner_radius, out_scale * outer_radius, num_sides)
  return target_curve, tube

In [ ]:
%matplotlib qt5


def plot_centerlines(
  centerlines,
  highlight_segments=(),
  lw=1.0,
  highlight_lw=3.0,
  dim_alpha=0.5,
  figsize=(4.0, 6.0),
  verbose=True,
):
  lines = split_lines(centerlines)
  highlight = set(highlight_segments)
  ax = plt.figure(figsize=figsize).add_subplot(projection="3d")
  for index, point_ids in enumerate(lines):
    points = centerlines.points[point_ids]
    length = np.linalg.norm(np.diff(points, axis=0), axis=-1).sum()
    if verbose:
      print(f"line {index}  length {length:.1f} mm")
    is_hl = index in highlight
    alpha = 1.0 if (is_hl or not highlight) else dim_alpha
    (line,) = ax.plot(
      points[:, 0],
      points[:, 1],
      points[:, 2],
      lw=highlight_lw if is_hl else lw,
      alpha=alpha,
      label=f"{index}",
    )
    mid = points[len(points) // 2]
    ax.text(
      mid[0],
      mid[1],
      mid[2],
      f"{index}",
      fontsize=8,
      ha="center",
      va="center",
      color=line.get_color(),
      alpha=alpha,
      fontweight="bold" if is_hl else "normal",
    )

  ax.set_aspect("equal")
  ax.set_axis_off()
  ax.legend(frameon=False)
  plt.show()
  return lines, ax


lines, ax = plot_centerlines(centerlines)

### Pick waypoints
Right-click (or press P) on a centerline to add a waypoint. It snaps to the nearest
centerline point. Press U to undo. Pick the mouth first. The spline through the
picks is drawn live, and the printed `waypoint_ids` can be pasted into the next cell.

In [ ]:
waypoint_ids = []
line_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

picker = pvqt.BackgroundPlotter(window_size=(1200, 900))
picker.add_mesh(
  lumen, color="firebrick", opacity=0.15, smooth_shading=True, pickable=False
)
for index, point_ids in enumerate(lines):
  line_points = centerlines.points[point_ids]
  picker.add_mesh(
    pv.lines_from_points(line_points),
    color=line_colors[index % len(line_colors)],
    line_width=4,
  )
  picker.add_point_labels(
    line_points[[len(line_points) // 2]],
    [f"{index}"],
    shape=None,
    always_visible=True,
    text_color=line_colors[index % len(line_colors)],
  )


def show_waypoints():
  picker.remove_actor(["waypoints", "spline"])
  if waypoint_ids:
    picker.add_mesh(
      pv.PolyData(centerlines.points[waypoint_ids]),
      color="k",
      point_size=12,
      render_points_as_spheres=True,
      name="waypoints",
      pickable=False,
    )
  if len(np.unique(centerlines.points[waypoint_ids], axis=0)) > 1:
    picker.add_mesh(
      pv.lines_from_points(waypoint_spline(centerlines, waypoint_ids, 0.25)),
      color="k",
      line_width=2,
      name="spline",
      pickable=False,
    )
  picker.add_text(f"waypoint_ids = {waypoint_ids}", font_size=10, name="waypoint_text")
  print(f"waypoint_ids = {waypoint_ids}")


def add_waypoint(point):
  waypoint_ids.append(int(point_tree.query(point)[1]))
  show_waypoints()


def undo_waypoint():
  if waypoint_ids:
    waypoint_ids.pop()
  show_waypoints()


picker.enable_point_picking(
  callback=add_waypoint,
  show_message="Right-click or P: add waypoint (mouth first),  U: undo",
  show_point=False,
)
picker.add_key_event("u", undo_waypoint)
picker.set_background("white")

### Build the target curve and the tube
Radii in mm. Outputs are in m, with the mouth at the origin and insertion along +x.

waypoint_ids = [1, 60, 114, 176, 229, 277, 304, 339, 368, 393, 4823, 3097, 4157, 4222, 3318, 5166, 2180, 4406, 3453]

In [ ]:
for i in (waypoint_ids):
  print(i, end=", ")

In [ ]:
# waypoint_ids = [...]  # paste here to rebuild without re-picking
waypoint_ids = [1, 60, 114, 176, 229, 277, 304, 339, 368, 393, 4823, 3097, 
                4157, 4222, 3318, 5166, 2180, 4406, 3453]
inner_radius, outer_radius = 1.5, 3.5  # mm, uniform catheter tube
spacing = 0.25  # mm

target_curve, tube = catheter_track(
  centerlines, waypoint_ids, inner_radius, outer_radius, spacing
)
segments = picked_lines(centerlines, lines, waypoint_ids)
open_edges = tube.extract_feature_edges(
  boundary_edges=True, feature_edges=False, manifold_edges=False
).n_cells
print(f"length {1e3 * target_curve.s.iloc[-1]:.1f} mm,  segments {segments}")
print(f"open edges {open_edges} (want 0)")

### Highlight the picked segments

In [ ]:
plot_centerlines(centerlines, highlight_segments=segments, verbose=False);

In [ ]:
context_color, section_color, context_opacity = "firebrick", "orange", 0.15
path = waypoint_spline(centerlines, waypoint_ids, spacing)
path_radius = radius[point_tree.query(path)[1]]

# Each lumen cell is assigned to its nearest point on the track. It counts as part
# of the track when that point is not an end of the track and the cell is within
# 1.5 lumen radii of it, which keeps walls of neighbouring branches out.
distance, nearest = spy_spatial.cKDTree(path).query(lumen.cell_centers().points)
in_section = (
  (nearest > 0)
  & (nearest < len(path) - 1)
  & (distance < 1.5 * path_radius[nearest])
)

plotter = pvqt.BackgroundPlotter(window_size=(1200, 900))
plotter.enable_depth_peeling()
plotter.add_mesh(
  lumen.extract_cells(np.flatnonzero(~in_section)),
  color=context_color,
  opacity=context_opacity,
  smooth_shading=True,
)
plotter.add_mesh(
  lumen.extract_cells(np.flatnonzero(in_section)),
  color=section_color,
  smooth_shading=True,
)
plotter.add_mesh(pv.lines_from_points(path), color="k", line_width=3)
plotter.set_background("white")

### Tube and target in the track frame

In [ ]:
plotter = pvqt.BackgroundPlotter(window_size=(1200, 900))
plotter.enable_depth_peeling()
plotter.add_mesh(tube, color="rosybrown", opacity=0.4, smooth_shading=True)
plotter.add_mesh(
  pv.lines_from_points(target_curve[["x", "y", "z"]].to_numpy()),
  color="k",
  line_width=3,
)
plotter.add_mesh(
  pv.PolyData(np.zeros((1, 3))),
  color="k",
  point_size=12,
  render_points_as_spheres=True,
)  # mouth
plotter.add_axes()
plotter.set_background("white")

### Save the target and the tube

In [ ]:
name = f"catheter_{case_dir}_{side}"#_{'_'.join(map(str, segments))}"
target_curve.to_csv(f"crops/data/targets/{name}.csv", index=False)
tube.save(f"crops/data/obstacles/{name}.stl")
print(f"saved crops/data/targets/{name}.csv and crops/data/obstacles/{name}.stl")